In [2]:
import os
from datetime import datetime
from collections import defaultdict

# folder path
folder_path = r"F:\strwaberry images\Front_row_with_text"

# store results
daily_stats = defaultdict(list)

# loop through files
for file in os.listdir(folder_path):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        try:
            # example: citra_2025-12-14_143001.jpg
            name = os.path.splitext(file)[0]   # remove .jpg
            parts = name.split("_")
            
            date_part = parts[1]              # 2025-12-14
            time_part = parts[2]              # 143001
            
            dt = datetime.strptime(
                f"{date_part} {time_part}",
                "%Y-%m-%d %H%M%S"
            )
            
            daily_stats[dt.date()].append(dt)
        
        except Exception as e:
            print(f"Skipping {file}: {e}")

# sort days
sorted_days = sorted(daily_stats.keys())

# overall stats
all_times = [t for v in daily_stats.values() for t in v]
print("Total Images:", len(all_times))
print("Start Date:", min(all_times))
print("End Date:", max(all_times))

print("\nDaily Statistics:\n")

for day in sorted_days:
    times = sorted(daily_stats[day])
    
    print(f"Date: {day}")
    print(f"Total Images: {len(times)}")
    # print(f"Start Time: {times[0].time()}")
    # print(f"End Time: {times[-1].time()}")
    print("-" * 40)

Total Images: 324
Start Date: 2025-11-27 13:34:01
End Date: 2025-12-21 11:15:01

Daily Statistics:

Date: 2025-11-27
Total Images: 2
----------------------------------------
Date: 2025-12-12
Total Images: 2
----------------------------------------
Date: 2025-12-13
Total Images: 37
----------------------------------------
Date: 2025-12-14
Total Images: 37
----------------------------------------
Date: 2025-12-15
Total Images: 37
----------------------------------------
Date: 2025-12-16
Total Images: 37
----------------------------------------
Date: 2025-12-17
Total Images: 37
----------------------------------------
Date: 2025-12-18
Total Images: 37
----------------------------------------
Date: 2025-12-19
Total Images: 38
----------------------------------------
Date: 2025-12-20
Total Images: 38
----------------------------------------
Date: 2025-12-21
Total Images: 22
----------------------------------------


In [2]:
import os
from PIL import Image

# input and output folders
input_folder = r"F:\strwaberry images\Front_row_with_text"
output_base = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front"

# cropping coordinates: (x1, y1, x2, y2)
crops = {
    # "down_1": (23,433 ,712,993),
    "top_1": (18,16, 712, 993),
    "top_3": (1362,47, 1982,506),
    "top_4": (2100, 189, 2458,480),
    "top_5": (2721,53, 3240,448),
}

# create output folders
for folder in crops.keys():
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# process images
for img_name in os.listdir(input_folder):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(input_folder, img_name)
        image = Image.open(img_path)

        for folder_name, (x1, y1, x2, y2) in crops.items():
            cropped = image.crop((x1, y1, x2, y2))

            save_path = os.path.join(output_base, folder_name, img_name)
            cropped.save(save_path)

print("Cropping completed")


Cropping completed


In [9]:
import cv2
import numpy as np
import os
from pathlib import Path

# -------- COLOR MATCH FUNCTION --------
def color_transfer(source, target):
    source = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    target = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")

    (lMeanSrc, aMeanSrc, bMeanSrc) = cv2.mean(source)[:3]
    (lStdSrc, aStdSrc, bStdSrc) = source.std(axis=(0,1))

    (lMeanTar, aMeanTar, bMeanTar) = cv2.mean(target)[:3]
    (lStdTar, aStdTar, bStdTar) = target.std(axis=(0,1))

    l, a, b = cv2.split(source)

    l = ((l - lMeanSrc) * (lStdTar / (lStdSrc+1e-6))) + lMeanTar
    a = ((a - aMeanSrc) * (aStdTar / (aStdSrc+1e-6))) + aMeanTar
    b = ((b - bMeanSrc) * (bStdTar / (bStdSrc+1e-6))) + bMeanTar

    result = cv2.merge([l, a, b])
    result = np.clip(result, 0, 255).astype("uint8")
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

# -------- LOAD REFERENCE --------
reference = cv2.imread(r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\top_5\citra_2025-12-17_074501.jpg")

input_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\top_5"     
output_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\top_5\preprocessed_images_lighting_removal"  
os.makedirs(output_folder, exist_ok=True)

# -------- PROCESS ALL IMAGES --------
for img_path in Path(input_folder).glob("*"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    normalized = color_transfer(img, reference)
    cv2.imwrite(os.path.join(output_folder, 'front_row_top5'+img_path.name), normalized)

print("✔ Dataset normalized to reference lighting")

✔ Dataset normalized to reference lighting


Renaming completed!


In [ ]:
import os

folder_path = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\down_1\preprocessed_images_lighting_removal"
prefix = "front_row_down1_"   # initial text you want to add

for filename in os.listdir(folder_path):
    old_path = os.path.join(folder_path, filename)

    # skip directories
    if not os.path.isfile(old_path):
        continue

    new_filename = prefix + filename
    new_path = os.path.join(folder_path, new_filename)

    os.rename(old_path, new_path)

print("Renaming completed!")

In [7]:
df = df.dropna(subset=["image_name"])
df.to_excel("updated_down1_front_row.xlsx", index=False)

In [31]:
##mapped weather with all the images
import os
import pandas as pd

# paths
excel_path = r"F:\strwaberry images\front_row_dataset\front_row_down1_down1_.xlsx"
image_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\down_1\preprocessed_images_lighting_removal"

# string to remove
remove_prefix = "front_row_down1_down1_"

# read excel
df = pd.read_excel(excel_path)

# make lookup dictionary
available_images = os.listdir(image_folder)

clean_map = {}
unmatched_folder_images = []

for img in available_images:
    if img.startswith(remove_prefix):
        cleaned = img.replace(remove_prefix, "")
        clean_map[cleaned] = img
    else:
        unmatched_folder_images.append(img)

# match with excel
new_names = []
missing_in_folder = []

for name in df["image_name"]:
    if name in clean_map:
        new_names.append(clean_map[name])
    else:
        new_names.append(None)
        missing_in_folder.append(name)

# add new column
df["new_image_name"] = new_names

# save output
output_path = f"{remove_prefix}.xlsx"
df.to_excel(output_path, index=False)

print("Matching completed")
print("Output saved:", output_path)

print("\nImages missing in folder:")
for m in missing_in_folder:
    print(m)

print("\nImages in folder but not used:")
unused = set(clean_map.values()) - set(df["new_image_name"].dropna())
for u in unused:
    print(u)

Matching completed
Output saved: front_row_down1_down1_.xlsx

Images missing in folder:

Images in folder but not used:
front_row_down1_down1_citra_2025-12-12_150001.jpg
front_row_down1_down1_front_row_donw_plant1.mp4
front_row_down1_down1_files.txt


In [30]:
import pandas as pd
import re
import os

excel_path = r"F:\strwaberry images\front_row_dataset\updated_down1_front_row.xlsx"

# read excel
df = pd.read_excel(excel_path)

# function to extract date & time
def extract_datetime(filename):
    match = re.search(r'citra_(\d{4}-\d{2}-\d{2})_(\d{6})', filename)
    if match:
        date = match.group(1)
        time_raw = match.group(2)
        time = f"{time_raw[:2]}:{time_raw[2:4]}:{time_raw[4:]}"
        return pd.Series([date, time])
    return pd.Series([None, None])

# create new columns
df[["date", "time"]] = df["new_image_name"].apply(extract_datetime)

# save with SAME name and SAME location
df.to_excel(excel_path, index=False)

print("Date and time columns added and file overwritten.")

KeyError: 'new_image_name'

In [57]:
##Green pixel mapping



import cv2
import numpy as np
import pandas as pd
from pathlib import Path


excel_path=r"F:\strwaberry images\front_row_dataset\front_row_top5.xlsx"
# load your main dataset
df_final = pd.read_excel(excel_path)

folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_front\top_5\preprocessed_images_lighting_removal"

green_dict = {}

for path in Path(folder).glob("*.jpg"):

    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_green = np.array([35,40,40])
    upper_green = np.array([90,255,255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    green_dict[path.name] = green_pixels


# add green_pixels column using image_name
df_final["green_pixels"] = df_final["new_image_name"].map(green_dict)
# df_final.to_excel(base_path+"\\"+excel_name, index=False)
df_final

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),image_name,new_image_name,date,time,green_pixels
0,260,1969-12-31 19:00:46.004,12.48,2.31,3.61,3.32,94.00,2.74,0,1.51,100.20,0.01,citra_2025-12-13_060001.jpg,front_row_top5citra_2025-12-13_060001.jpg,2025-12-13,06:00:01,42895
1,260,1969-12-31 19:00:46.004,12.43,2.39,3.44,3.28,94.00,2.57,0,1.45,92.80,0.02,citra_2025-12-13_061501.jpg,front_row_top5citra_2025-12-13_061501.jpg,2025-12-13,06:15:01,37677
2,260,1969-12-31 19:00:46.004,12.38,2.09,3.40,3.28,94.00,2.53,0,0.98,87.50,0.01,citra_2025-12-13_063001.jpg,front_row_top5citra_2025-12-13_063001.jpg,2025-12-13,06:30:01,36504
3,260,1969-12-31 19:00:46.004,12.33,2.02,3.36,3.34,94.00,2.49,0,1.22,100.10,0.02,citra_2025-12-13_064501.jpg,front_row_top5citra_2025-12-13_064501.jpg,2025-12-13,06:45:01,35484
4,260,1969-12-31 19:00:46.004,12.29,2.04,3.24,3.42,94.00,2.37,0,1.07,75.38,0.18,citra_2025-12-13_070001.jpg,front_row_top5citra_2025-12-13_070001.jpg,2025-12-13,07:00:01,35577
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,260,1969-12-31 19:00:46.012,12.44,14.03,15.15,13.73,86.50,12.91,0,1.82,41.25,460.30,citra_2025-12-21_101501.jpg,front_row_top5citra_2025-12-21_101501.jpg,2025-12-21,10:15:01,26298
316,260,1969-12-31 19:00:46.012,12.46,16.09,16.92,15.31,68.58,11.11,0,3.51,31.14,486.90,citra_2025-12-21_103001.jpg,front_row_top5citra_2025-12-21_103001.jpg,2025-12-21,10:30:01,33535
317,260,1969-12-31 19:00:46.012,12.49,17.40,17.72,16.29,65.73,11.23,0,4.46,35.11,508.20,citra_2025-12-21_104501.jpg,front_row_top5citra_2025-12-21_104501.jpg,2025-12-21,10:45:01,27423
318,260,1969-12-31 19:00:46.012,12.54,18.48,18.97,17.27,62.68,11.70,0,3.78,44.05,532.50,citra_2025-12-21_110001.jpg,front_row_top5citra_2025-12-21_110001.jpg,2025-12-21,11:00:01,28207


In [58]:
##Gdd mapping

import pandas as pd
import re
# base_path='F:\strwaberry images\front_row_dataset'
# excel_name="front_row_down1_down1_.xlsx"
# paths
# excel_path = r"F:\strwaberry images\front_row_dataset\front_row_down1_down1_.xlsx"
weather_excel = r"F:\strwaberry images\Back_row_excel_dataset\final_dataset_back_setup_top_plant5.xlsx"   # file that has Date, Tmax, Tmin, GDD

# read main excel
df = pd.read_excel(excel_path)

# -------- extract date & time from new_image_name ----------
def extract_datetime(filename):
    match = re.search(r'citra_(\d{4}-\d{2}-\d{2})_(\d{6})', str(filename))
    if match:
        date = match.group(1)
        t = match.group(2)
        time = f"{t[:2]}:{t[2:4]}:{t[4:]}"
        return pd.Series([date, time])
    return pd.Series([None, None])

df[["date", "time"]] = df["new_image_name"].apply(extract_datetime)

# -------- read green pixel weather file ----------
weather = pd.read_excel(weather_excel)

# expected columns in weather file:
# date | Tmax (Temp @ 2m) | Tmin (Temp @ 2m) | GDD(Tbase=7)

# make sure date format matches
df["date"] = pd.to_datetime(df["date"]).dt.date
weather["date"] = pd.to_datetime(weather["date"]).dt.date

# -------- merge by date ----------
df = df.merge(
    weather[["date", "Tmax (Temp @ 2m)", "Tmin (Temp @ 2m)", "GDD(Tbase=7)"]],
    on="date",
    how="left"
)
df
# save with same name
df.to_excel(excel_path, index=False)

print("Date, time, Tmax, Tmin, GDD added successfully")

Date, time, Tmax, Tmin, GDD added successfully


In [59]:
##merge all the excels of the the folder into one excel
import pandas as pd
import os
from glob import glob

# folder containing multiple excel files
folder_path = r"F:\strwaberry images\front_row_dataset"

# output merged file
output_file = os.path.join(folder_path, "Front_row_merged_outpt.xlsx")

# get all excel files
excel_files = glob(os.path.join(folder_path, "*.xlsx"))

# remove temp files like ~$file.xlsx
excel_files = [f for f in excel_files if not os.path.basename(f).startswith("~$")]

# read and merge
df_list = [pd.read_excel(file) for file in excel_files]
merged_df = pd.concat(df_list, ignore_index=True)

# save merged file
merged_df.to_excel(output_file, index=False)

print("Merged file saved at:", output_file)

Merged file saved at: F:\strwaberry images\front_row_dataset\Front_row_merged_outpt.xlsx


In [61]:
import pandas as pd

excel_path = r"F:\strwaberry images\front_row_dataset\Front_row_merged_outpt.xlsx"   # your final merged excel

df = pd.read_excel(excel_path)

# mapping based on prefix inside new_image_name
plant_map = {
    "front_row_down1": 21,
    "front_row_down2": 22,
    "front_row_down3": 23,
    "front_row_down4": 24,
    "front_row_down5": 25,
    "front_row_top1": 11,
    "front_row_top3": 13,
    "front_row_top4": 14,
    "front_row_top5": 15
}

def assign_plant_id(name):
    name = str(name)
    for key, value in plant_map.items():
        if key in name:
            return value
    return None

# create new column
df["plant_id"] = df["new_image_name"].apply(assign_plant_id)

# save back
df.to_excel(excel_path, index=False)
df
print("plant_id column added")

plant_id column added
